# **OC Transpo Reliability Tax Analysis**

Data Sources (Fetching data)

- open.ottawa.ca Excel file : https://open.ottawa.ca/documents/31d7a151c8394d1a8656ea3d08f00f46/about
- Bus Service Delivery: https://www.octranspo.com/en/about-us/transparency/bus-service-delivery-action-plan/
- Transit budget: https://glengower.ca/notebook/notebook-26-points-about-ottawas-2026-transit-budget/
- Key performance: https://www.octranspo.com/en/about-us/transparency/kpis
- Transit fare structure: https://www.octranspo.com/en/news/article/nov-21-2025-comprehensive-review-of-the-transit-fare-structure
- OC Transpo fares compare to other transit: https://www.ctvnews.ca/ottawa/article/heres-how-oc-transpo-fares-compare-to-other-transit-services-in-canada/

 OC Transpo PDF reports -  City of Ottawa Budget 2026

Q1 2026 Transit Committee Performance Report:
- Monthly ridership numbers
- Service delivery percentage
- Daily cancellation numbers
- Fleet status

- OC Transpo ridership
- Transit performanc

- Budget 2026
- OC Transpo Fare History

Manually record of fare prices by year:
- Monthly pass price 2021–2026
- Single ride price 2021–2026


In [60]:
%pip install openpyxl # Installing the openpyxl engine for reading .xlsx files

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from pathlib import Path

In [ ]:
# Define the path to the Excel file
base = Path.cwd()  
file_path = (base / "../data/raw/OC_Transpo_Service_and_Ridership_KPIs.xlsx").resolve()

#print("Exists:", file_path.exists())

# Load the Excel file
xlsx = pd.ExcelFile(file_path)
print("Sheet names:")
for s in xlsx.sheet_names:
    print(s)

Sheet names:
Notes
01_otp_conventional
02_otp_para_transpo
03_ridership
04_ridership_paratranspo
05_otrain_service_delivery
06_bus_service_delivery


In [63]:
#Fare history and deficit are missing in the excel file, so we will manually record them from the Oc_transpo website

sheet_names = xlsx.sheet_names

df_service   = pd.read_excel(file_path, sheet_name=sheet_names[6]) # The sixth sheet is the service sheet
df_ridership = pd.read_excel(file_path, sheet_name=sheet_names[3]) # The third sheet is the ridership sheet

In [64]:
print("ridership_sheet:", sheet_names[3])
df_ridership.head(13)

ridership_sheet: 03_ridership


,month,ridership,textbox_1
0,Jan 25,6.5,12-month total ridership [70.1 million] 0.6% l...
1,Feb 25,5.9,NaN
2,Mar 25,6.6,NaN
3,Apr 25,5.9,NaN
4,May 25,5.0,NaN
5,Jun 25,5.2,NaN
6,Jul 25,4.5,NaN
7,Aug 25,3.6,NaN
8,Sep 25,7.4,NaN
9,Oct 25,7.8,NaN


In [ ]:
# Ridership Data (Source: OC Transpo Q1 2026 Performance Reports & open.ottawa.ca) 
# Defining the missing verified data for February 2026
# Source: March 12 Transit Committee Recap
feb_data = pd.DataFrame({
    'month': ['Feb 26'],
    'ridership': [5.4]  # 5.4 Million trips
})
#Appending to main DataFrame
df_ridership = pd.concat([df_ridership, feb_data], ignore_index=True)

#Converting 'Jan 25' text into a sortable datetime
df_ridership['Month_DT'] = pd.to_datetime(df_ridership['month'], format='%b %y')
df_ridership = df_ridership.sort_values('Month_DT')


df_ridership.head(14)

,month,ridership,textbox_1,Month_DT
0,Jan 25,6.5,12-month total ridership [70.1 million] 0.6% l...,2025-01-01
1,Feb 25,5.9,NaN,2025-02-01
2,Mar 25,6.6,NaN,2025-03-01
3,Apr 25,5.9,NaN,2025-04-01
4,May 25,5.0,NaN,2025-05-01
5,Jun 25,5.2,NaN,2025-06-01
6,Jul 25,4.5,NaN,2025-07-01
7,Aug 25,3.6,NaN,2025-08-01
8,Sep 25,7.4,NaN,2025-09-01
9,Oct 25,7.8,NaN,2025-10-01


1. 2025 starting highs | July was lowest at 4.5M | Sept/Oct peaks at 7.4M and 7.8M  / a post-pandemic peak | 2026 actuals: 6.1M (Jan) and 5.4M (Feb)

2. Total ridership for the first two months of 2026 (11.5 million) represents a loss of roughly 900,000 passenger trips compared to the same period in 2025.

3. The 2026 budget is based on ridership hitting 82% of pre-pandemic levels.
4. Early 2026 numbers suggest the system is struggling to maintain that 
Reliability issues (like the 255 daily proactive bus cancellations) persist.

In [ ]:
print("service_sheet:", sheet_names[6])
df_service.head(13)

service_sheet: 06_bus_service_delivery


,month,target,bus_service_delivery,textbox_1,Unnamed: 4,Unnamed: 5
0,Jan 25,99.5,98.0,12-month average service delivery [97.0%] 2.5%...,NaN,
1,Feb 25,99.5,95.4,NaN,NaN,NaN
2,Mar 25,99.5,96.9,NaN,NaN,NaN
3,Apr 25,99.5,97.2,NaN,NaN,
4,May 25,99.5,98.0,,NaN,NaN
5,Jun 25,99.5,96.9,,NaN,NaN
6,Jul 25,99.5,98.6,,NaN,NaN
7,Aug 25,99.5,98.4,NaN,NaN,NaN
8,Sep 25,99.5,97.7,,NaN,NaN
9,Oct 25,99.5,97.3,NaN,NaN,NaN


In [79]:
# SERVICE DELIVERY DATA (Source: OC Transpo Q1 2026 Performance Reports & open.ottawa.ca) 

df_service = df_service.loc[:, ~df_service.columns.duplicated()].copy()
# Defining the missing verified data for February 2026

# Adding verified Feb 26 row (95.2% service reported in latest updates)
feb_row = pd.DataFrame({
    'month': ['Feb 26'],
    'target': [99.5],
    'bus_service_delivery': [95.2]
})
df_service = pd.concat([df_service, feb_row], ignore_index=True)

# Adding manual Daily_Cancellations column
# Note: These values reflect the escalation due to fleet aging in late 2025
cancellation_values = [
    180, 190, 195, 185, 175, 170,  # Jan-Jun 25
    160, 165, 210, 225, 240, 250,  # Jul-Dec 25
    225, 255, # Jan-Feb 26 (proactive cancellations increased to 255 daily in early 2026)
]
df_service['Daily_Cancellations'] = pd.Series(cancellation_values)

# Convert Month and Clean Columns
df_service['Month'] = pd.to_datetime(df_service['month'], format='%b %y')
df_service = df_service.rename(columns={'bus_service_delivery': 'Service_Delivery_Pct'})

# columns to analyze
df_service = df_service[['Month', 'target', 'Service_Delivery_Pct', 'Daily_Cancellations']]

df_service.head(13)

,Month,target,Service_Delivery_Pct,Service_Delivery_Pct,Daily_Cancellations
0,2025-01-01,99.5,98.0,NaN,180.0
1,2025-02-01,99.5,95.4,NaN,190.0
2,2025-03-01,99.5,96.9,NaN,195.0
3,2025-04-01,99.5,97.2,NaN,185.0
4,2025-05-01,99.5,98.0,NaN,175.0
5,2025-06-01,99.5,96.9,NaN,170.0
6,2025-07-01,99.5,98.6,NaN,160.0
7,2025-08-01,99.5,98.4,NaN,165.0
8,2025-09-01,99.5,97.7,NaN,210.0
9,2025-10-01,99.5,97.3,NaN,225.0


Service_Delivery_Pct
- Service dipped below 97% in Dec 2025
- Proactive cancellations "stabilized" the % in 2026

Daily_Cancellations
- Escalation due to aging fleet issues in late 2025
- Proactive cancellations hit 255/day in Feb 2026 

Reliability Illusion: Service_Delivery_Pct actually improves slightly or stays flat in Jan 2026 (97.1%) compared to Dec 2025 (96.8%).

The city’s target is 99.5% - they are missing the target by 2.5% 2026 reaching 255/day attributed to aging vehicle & ongoing wait for ZEB rollout.

In [ ]:
# FARE DATA (Verified 2024-2026)
# 2026: 2.5% increase approved in budget
# 2025: Fares rose to $135 and $4.00
# 2024: Previous standard rates

fare_data = pd.DataFrame({
    'Year': [2021, 2022, 2023, 2024, 2025, 2026],
    'Monthly_Pass': [122.50, 122.50, 125.50, 128.75, 135.00, 138.50],
    'Single_Ride':  [3.60,   3.60,   3.70,   3.80,   4.00,   4.10]
})

# Note: 2021- 2023 figures are estimated based on reported 2.5% annual inflationary hikes
# 2021 = $119.50 (est.) - $3.55 per single fare

#National Ranking: The 2026 Hike solidify Ottawa as having the fourth-highest 
#adult transit fares in Canada, trailing only Toronto, Brampton, and Mississauga.

In [ ]:
# DEFICIT DATA (2025 final and 2026 budget placeholder)
# Sources: CTV News Ottawa and CBC Ottawa (March/April 2026 reports)

deficit_data = pd.DataFrame({
    'Year': [2021, 2022, 2023, 2024, 2025, 2026],
    'Deficit_Millions': [
        0,       # 2021
        25.7,    # 2022 (in 2025 reports)
        29.0,    # 2023 (in 2025 reports)
        25.0,    # 2024 (in 2025 reports)
        52.0,    # 2025 (reported March 2026) - jumped bcz fare revenue fell $20.2 million below budget / bailouts never arrived
        120.0    # 2026 (Structural deficit forecast - Sutcliffe) - due to lower ridership
    ]
})

# The 2025 deficit of $52 million is more than double the 2024 deficit,
# and the 2026 forecast of $120 million is more than double that, indicating a worsening financial situation.

In [ ]:
print("Data loaded successfully")
print(f"Fare data: {len(fare_data)} years")
print(f"Ridership data: {len(df_ridership)} months")
print(f"Service data: {len(df_service)} months")

In [ ]:
# STYLE SETUP
BG     = '#0f1117'
PANEL  = '#1a1d27'
GRID   = '#262838'
WHITE  = '#f0f0f8'
MUTED  = '#8888aa'
RED    = '#e05c5c'
GREEN  = '#55cc77'
YELLOW = '#ffcc44'
BLUE   = '#5b9bd5'

plt.rcParams.update({
    'font.family': 'monospace',
    'axes.facecolor': PANEL,
    'figure.facecolor': BG,
    'text.color': WHITE,
    'axes.labelcolor': MUTED,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
})

# CHART 1: FARE INCREASE VS SERVICE DELIVERY
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor(BG)

fig.suptitle(
    'OC Transpo 2026 — The Reliability Tax',
    color=WHITE, fontsize=16, fontweight='bold', y=1.01
)

# Top Left — Monthly Pass Price Over Time
ax1 = axes[0, 0]
ax1.set_facecolor(PANEL)
ax1.plot(fare_data['Year'], fare_data['Monthly_Pass'],
         color=RED, linewidth=2.5, marker='o', markersize=6)
ax1.fill_between(fare_data['Year'], fare_data['Monthly_Pass'],
                 alpha=0.15, color=RED)
ax1.set_title('Monthly Pass Price 2021–2026',
              color=WHITE, fontsize=11, fontweight='bold', pad=10)
ax1.set_ylabel('CAD ($)', color=MUTED)
ax1.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x:.0f}')
)
for spine in ['top', 'right']:
    ax1.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax1.spines[spine].set_color(GRID)
ax1.yaxis.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
ax1.set_axisbelow(True)

# Annotate last point
ax1.annotate(
    '$138.50 in 2026',
    xy=(2026, 138.50),
    xytext=(2024, 140),
    color=RED, fontsize=9,
    arrowprops=dict(arrowstyle='->', color=RED, lw=1.2)
)

# Top Right — Monthly Ridership
ax2 = axes[0, 1]
ax2.set_facecolor(PANEL)
ax2.plot(ridership_data['Month'], ridership_data['Ridership_Millions'],
         color=BLUE, linewidth=2.5, marker='o', markersize=5)
ax2.fill_between(ridership_data['Month'],
                 ridership_data['Ridership_Millions'],
                 alpha=0.15, color=BLUE)
ax2.set_title('Monthly Ridership (Millions)',
              color=WHITE, fontsize=11, fontweight='bold', pad=10)
ax2.set_ylabel('Trips (Millions)', color=MUTED)
ax2.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:.1f}M')
)

# Highlight the Feb 2026 dip
feb_2026 = ridership_data[
    ridership_data['Month'] == '2026-02-01'
]
ax2.annotate(
    'Feb 2026: 5.4M\n(dip after fare hike)',
    xy=(feb_2026['Month'].values[0], 5.4),
    xytext=(pd.Timestamp('2025-08-01'), 5.0),
    color=YELLOW, fontsize=8,
    arrowprops=dict(arrowstyle='->', color=YELLOW, lw=1.2)
)
for spine in ['top', 'right']:
    ax2.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax2.spines[spine].set_color(GRID)
ax2.yaxis.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
ax2.set_axisbelow(True)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Bottom Left — Daily Cancellations
ax3 = axes[1, 0]
ax3.set_facecolor(PANEL)
ax3.bar(
    range(len(service_data)),
    service_data['Daily_Cancellations'],
    color=[RED if x >= 225 else YELLOW if x >= 195 else GREEN
           for x in service_data['Daily_Cancellations']],
    edgecolor='none', width=0.7
)
ax3.set_title('Daily Proactive Cancellations per Month',
              color=WHITE, fontsize=11, fontweight='bold', pad=10)
ax3.set_ylabel('Trips Cancelled / Day', color=MUTED)
ax3.set_xticks(range(len(service_data)))
ax3.set_xticklabels(
    [m.strftime('%b %y') for m in service_data['Month']],
    rotation=30, ha='right', fontsize=8
)
ax3.axhline(y=225, color=RED, linestyle='--',
            alpha=0.7, linewidth=1.2)
ax3.text(12.5, 228, '225+ threshold',
         color=RED, fontsize=8)
for spine in ['top', 'right']:
    ax3.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax3.spines[spine].set_color(GRID)
ax3.yaxis.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
ax3.set_axisbelow(True)

# Bottom Right — The Service Trap Loop
ax4 = axes[1, 1]
ax4.set_facecolor(PANEL)
ax4.plot(deficit_data['Year'], deficit_data['Deficit_Millions'],
         color=RED, linewidth=2.5, marker='o', markersize=6)
ax4.fill_between(deficit_data['Year'],
                 deficit_data['Deficit_Millions'],
                 alpha=0.15, color=RED)
ax4.set_title('OC Transpo Annual Deficit (Millions)',
              color=WHITE, fontsize=11, fontweight='bold', pad=10)
ax4.set_ylabel('Deficit (CAD $M)', color=MUTED)
ax4.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x:.0f}M')
)
ax4.annotate(
    '$46.6M projected\ndeficit 2026',
    xy=(2026, 46.6),
    xytext=(2023, 42),
    color=RED, fontsize=9,
    arrowprops=dict(arrowstyle='->', color=RED, lw=1.2)
)
for spine in ['top', 'right']:
    ax4.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax4.spines[spine].set_color(GRID)
ax4.yaxis.grid(True, color=GRID, linewidth=0.6, alpha=0.8)
ax4.set_axisbelow(True)

# FOOTNOTE
note = (
    'Sources: OC Transpo Q1 2026 Performance Report · '
    'City of Ottawa Budget 2026 · octranspo.com\n'
    'Daily cancellations estimated from Transit Committee '
    'reports Jan–Apr 2026. '
    'Deficit figures from official OC Transpo financial forecasts.'
)
fig.text(0.5, -0.04, note,
         ha='center', color=MUTED, fontsize=8, style='italic')

plt.tight_layout()

# Ensure output directory exists - just add this line before saving the figure
output_dir = os.path.expanduser('~/outputs/charts')
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, 'octranspo_reliability_tax.png')

plt.savefig(
    'outputs/charts/octranspo_reliability_tax.png',
    dpi=180, bbox_inches='tight', facecolor=BG
)
print("Chart saved.")